In [8]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [9]:
ADR_PERIOD = 5

RTH_START = "09:30"
RTH_END = "16:15"

In [5]:
pd.read_csv("../data/nq-1m_bk.csv", nrows=5).columns

Index(['11/12/2008;01:38;1416.299899;1416.299899;1416.299899;1416.299899;1'], dtype='str')

In [10]:
DATA_PATH = Path("../data/nq-1m_bk.csv")

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    header=None,
    names=[
        "Date",
        "Time",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
    ],
)

df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M",
)

df = (
    df.drop(columns=["Date", "Time"])
      .sort_values("Datetime")
      .reset_index(drop=True)
)

df = df[
    [
        "Datetime",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
    ]
]

print(f"Rows loaded: {len(df):,}")

Rows loaded: 5,884,752


In [11]:
df = df[
    df["Datetime"] >= "2022-07-01"
].reset_index(drop=True)

print(f"Rows loaded: {len(df):,}")

Rows loaded: 1,414,163


In [12]:
df["SessionDate"] = (
    df["Datetime"] - pd.Timedelta(hours=7)
).dt.floor("D")

rth = df[
    (df["Datetime"].dt.time >= pd.Timestamp("09:30").time()) &
    (df["Datetime"].dt.time <= pd.Timestamp("16:15").time())
].copy()

print(f"Sessions: {rth['SessionDate'].nunique()}")

Sessions: 1030


In [13]:
daily = (
    rth.groupby("SessionDate")
       .agg(
           High=("High", "max"),
           Low=("Low", "min"),
       )
)

daily["Range"] = daily["High"] - daily["Low"]

daily["ADR5"] = (
    daily["Range"]
    .rolling(ADR_PERIOD)
    .mean()
    .shift(1)
)

daily.head(10)

,High,Low,Range,ADR5
SessionDate,,,,
2022-07-01,13671.940798,13403.022166,268.918632,NaN
2022-07-04,13606.988866,13558.201442,48.787424,NaN
2022-07-05,13890.602505,13481.493505,409.109000,NaN
2022-07-06,14076.935197,13817.127469,259.807728,NaN
2022-07-07,14305.295608,14122.195819,183.099789,NaN
2022-07-08,14355.258633,14152.467534,202.791099,233.944515
2022-07-11,14112.791014,13940.271630,172.519384,220.719008
2022-07-12,14039.022078,13768.633946,270.388132,245.465400
2022-07-13,13936.157028,13694.865009,241.292019,217.721226


In [14]:
daily["PrevClose"] = (
    rth.groupby("SessionDate")["Close"]
       .last()
       .shift(1)
)

daily["Open"] = (
    rth.groupby("SessionDate")["Open"]
       .first()
)

daily.head()

,High,Low,Range,ADR5,PrevClose,Open
SessionDate,,,,,,
2022-07-01,13671.940798,13403.022166,268.918632,NaN,NaN,13459.157094
2022-07-04,13606.988866,13558.201442,48.787424,NaN,13649.310487,13567.018447
2022-07-05,13890.602505,13481.493505,409.109000,NaN,13582.889055,13521.170024
2022-07-06,14076.935197,13817.127469,259.807728,NaN,13885.606203,13895.304908
2022-07-07,14305.295608,14122.195819,183.099789,NaN,13992.585855,14148.646832


In [18]:
daily["Gap"] = daily["Open"] - daily["PrevClose"]

daily["GapDirection"] = np.where(
    daily["Gap"] > 0,
    "Up",
    np.where(daily["Gap"] < 0, "Down", "Flat")
)

daily["GapADR"] = daily["Gap"].abs() / daily["ADR5"]

daily[["PrevClose", "Open", "Gap", "GapDirection", "ADR5", "GapADR"]].tail()

,PrevClose,Open,Gap,GapDirection,ADR5,GapADR
SessionDate,,,,,,
2026-06-24,29752.00,29705.25,-46.75,Down,332.40,0.140644
2026-06-25,30095.25,29586.00,-509.25,Down,391.60,1.300434
2026-06-26,29759.25,29593.50,-165.75,Down,419.65,0.394972
2026-06-29,29283.00,29544.25,261.25,Up,498.85,0.523705
2026-06-30,29995.00,30410.75,415.75,Up,561.45,0.740493


In [41]:
gap_bins = [0.0, 0.5, np.inf]

daily["GapBucket"] = pd.cut(
    daily["GapADR"],
    bins=gap_bins,
    labels=[
        "< 0.5 ADR",
        "≥ 0.5 ADR",
    ],
    right=False,
    include_lowest=True,
)

In [54]:
results = []

TARGET_MULTIPLE = 0.05

for session_date, row in daily.dropna(subset=["ADR5", "PrevClose"]).iterrows():

    session = rth[rth["SessionDate"] == session_date]

    if session.empty:
        continue

    open_price = row["Open"]
    target = row["ADR5"] * TARGET_MULTIPLE

    if row["GapDirection"] == "Up":
        continuation = open_price + target
        reversal = open_price - target
    elif row["GapDirection"] == "Down":
        continuation = open_price - target
        reversal = open_price + target
    else:
        continue

    outcome = "Neither"

    for _, candle in session.iterrows():
        high = candle["High"]
        low = candle["Low"]

        if row["GapDirection"] == "Up":
            continuation_hit = high >= continuation
            reversal_hit = low <= reversal
        else:
            continuation_hit = low <= continuation
            reversal_hit = high >= reversal

        if continuation_hit and reversal_hit:
            outcome = "Unknown"
            break
        elif continuation_hit:
            outcome = "Continuation"
            break
        elif reversal_hit:
            outcome = "Reversal"
            break

    results.append({
        "SessionDate": session_date,
        "GapADR": row["GapADR"],
        "GapBucket": row["GapBucket"],
        "GapDirection": row["GapDirection"],
        "Outcome": outcome,
    })

results = pd.DataFrame(results)

results.head()

,SessionDate,GapADR,GapBucket,GapDirection,Outcome
0,2022-07-08,0.011307,< 0.5 ADR,Down,Reversal
1,2022-07-11,1.314248,≥ 0.5 ADR,Down,Reversal
2,2022-07-12,0.284962,< 0.5 ADR,Down,Reversal
3,2022-07-13,0.614201,≥ 0.5 ADR,Down,Continuation
4,2022-07-14,0.977753,≥ 0.5 ADR,Down,Continuation


In [55]:
summary = (
    results.groupby(["GapBucket", "Outcome"])
    .size()
    .unstack(fill_value=0)
)

summary["Total"] = summary.sum(axis=1)

for col in ["Continuation", "Reversal", "Unknown", "Neither"]:
    if col not in summary.columns:
        summary[col] = 0

display_table = pd.DataFrame(index=summary.index)

for col in ["Continuation", "Reversal", "Unknown", "Neither"]:
    pct = 100 * summary[col] / summary["Total"]
    display_table[col] = (
        summary[col].astype(int).astype(str)
        + " ("
        + pct.round(1).astype(str)
        + "%)"
    )

display_table["Total"] = summary["Total"]

overall = pd.DataFrame(summary.sum()).T
overall.index = ["All"]

overall["Total"] = summary["Total"].sum()

for col in ["Continuation", "Reversal", "Unknown", "Neither"]:
    pct = 100 * summary[col].sum() / summary["Total"].sum()
    overall[col] = f'{summary[col].sum()} ({pct:.1f}%)'

display_table = pd.concat([display_table, overall])

display_table

,Continuation,Reversal,Unknown,Neither,Total
< 0.5 ADR,284 (51.7%),246 (44.8%),18 (3.3%),1 (0.2%),549
≥ 0.5 ADR,229 (48.1%),227 (47.7%),20 (4.2%),0 (0.0%),476
All,513 (50.0%),473 (46.1%),38 (3.7%),1 (0.1%),1025


In [56]:
results = []

for session_date, row in daily.dropna(subset=["PrevClose"]).iterrows():

    if row["GapDirection"] == "Flat":
        continue

    session = rth[rth["SessionDate"] == session_date].iloc[:60]

    if session.empty:
        continue

    gap = abs(row["Gap"])
    fill_level = (
        row["Open"] - 0.5 * gap
        if row["GapDirection"] == "Up"
        else row["Open"] + 0.5 * gap
    )

    if row["GapDirection"] == "Up":
        filled = (session["Low"] <= fill_level).any()
    else:
        filled = (session["High"] >= fill_level).any()

    results.append({
        "SessionDate": session_date,
        "GapDirection": row["GapDirection"],
        "Gap": row["Gap"],
        "GapADR": row["GapADR"],
        "Filled50": filled,
    })

results = pd.DataFrame(results)

summary = (
    results["Filled50"]
    .value_counts()
    .rename(index={True: "Filled", False: "Not Filled"})
    .to_frame("Count")
)

summary["Percent"] = (
    100 * summary["Count"] / summary["Count"].sum()
).round(1)

summary

,Count,Percent
Filled50,,
Not Filled,538,52.3
Filled,491,47.7
